# Circuit

> SAX Circuits

In [ ]:
import gdsfactory as gf
import kfactory as kf
from gdsfactory.gpdk import PDK
from kfnetlist import HierarchicalNetlist
from kfnetlist.extract import extract

import sax
from sax.circuits import draw_dag

PDK.activate()

def native_netlist(component):
    return HierarchicalNetlist(extract(
        component,
        wrap_kdb_instance=lambda inst: kf.Instance(kcl=component.kcl, instance=inst),
        include_placement=False,
    ))

Let's start by creating a simple recursive netlist with gdsfactory.


```{note}
We are using gdsfactory to create our netlist because it allows us to see the circuit we want to simulate and because we're striving to have a compatible netlist implementation in SAX.
 
 However... gdsfactory is not a dependency of SAX. You can also define your circuits by hand (see SAX Quick Start Notebook) or you can use another tool to programmatically construct your netlists.
 ```

In [ ]:
@gf.cell
def mzi(delta_length=10.0):
    c = gf.Component()

    # components
    mmi_in = gf.components.mmi1x2()
    mmi_out = gf.components.mmi2x2()
    bend = gf.components.bend_euler()
    half_delay_straight = gf.components.straight(length=delta_length / 2.0)

    # references
    mmi_in = c.add_ref(mmi_in, name="mmi_in")
    mmi_out = c.add_ref(mmi_out, name="mmi_out")
    straight_top1 = c.add_ref(half_delay_straight, name="straight_top1")
    straight_top2 = c.add_ref(half_delay_straight, name="straight_top2")
    bend_top1 = c.add_ref(bend, name="bend_top1")
    bend_top2 = c.add_ref(bend, name="bend_top2").dmirror()
    bend_top3 = c.add_ref(bend, name="bend_top3").dmirror()
    bend_top4 = c.add_ref(bend, name="bend_top4")
    bend_btm1 = c.add_ref(bend, name="bend_btm1").dmirror()
    bend_btm2 = c.add_ref(bend, name="bend_btm2")
    bend_btm3 = c.add_ref(bend, name="bend_btm3")
    bend_btm4 = c.add_ref(bend, name="bend_btm4").dmirror()

    # connections
    bend_top1.connect("o1", mmi_in.ports["o2"])
    straight_top1.connect("o1", bend_top1.ports["o2"])
    bend_top2.connect("o1", straight_top1.ports["o2"])
    bend_top3.connect("o1", bend_top2.ports["o2"])
    straight_top2.connect("o1", bend_top3.ports["o2"])
    bend_top4.connect("o1", straight_top2.ports["o2"])

    bend_btm1.connect("o1", mmi_in.ports["o3"])
    bend_btm2.connect("o1", bend_btm1.ports["o2"])
    bend_btm3.connect("o1", bend_btm2.ports["o2"])
    bend_btm4.connect("o1", bend_btm3.ports["o2"])

    mmi_out.connect("o1", bend_btm4.ports["o2"])

    # ports
    c.add_port(
        "o1",
        port=mmi_in.ports["o1"],
    )
    c.add_port("o2", port=mmi_out.ports["o3"])
    c.add_port("o3", port=mmi_out.ports["o4"])
    return c


@gf.cell
def twomzi():
    c = gf.Component()

    # instances
    mzi1 = mzi(delta_length=10)
    mzi2 = mzi(delta_length=20)

    # references
    mzi1_ = c.add_ref(mzi1, name="mzi1")
    mzi2_ = c.add_ref(mzi2, name="mzi2")

    # connections
    mzi2_.connect("o1", mzi1_.ports["o2"])

    # ports
    c.add_port("o1", port=mzi1_.ports["o1"])
    c.add_port("o2", port=mzi2_.ports["o2"])
    return c

In [ ]:
comp = twomzi()
comp

In [ ]:
recnet = native_netlist(comp)
mzi1_comp = recnet[comp.name].instances["mzi1"].netlist_id
flatnet = recnet[mzi1_comp]

To be able to model this device we'll need some SAX dummy models:

In [ ]:
def bend_euler(
    angle=90.0,
    p=0.5,
    # cross_section="strip",
    # direction="ccw",
    # with_bbox=True,
    # with_arc_floorplan=True,
    # npoints=720,
):
    return sax.reciprocal({("o1", "o2"): 1.0})

In [ ]:
def mmi1x2(
    width=0.5,
    width_taper=1.0,
    length_taper=10.0,
    length_mmi=5.5,
    width_mmi=2.5,
    gap_mmi=0.25,
    # cross_section= strip,
    # taper= {function= taper},
    # with_bbox= True,
):
    return sax.reciprocal(
        {
            ("o1", "o2"): 0.45**0.5,
            ("o1", "o3"): 0.45**0.5,
        }
    )

In [ ]:
def mmi2x2(
    width=0.5,
    width_taper=1.0,
    length_taper=10.0,
    length_mmi=5.5,
    width_mmi=2.5,
    gap_mmi=0.25,
    # cross_section= strip,
    # taper= {function= taper},
    # with_bbox= True,
):
    return sax.reciprocal(
        {
            ("o1", "o3"): 0.45**0.5,
            ("o1", "o4"): 1j * 0.45**0.5,
            ("o2", "o3"): 1j * 0.45**0.5,
            ("o2", "o4"): 0.45**0.5,
        }
    )

In [ ]:
def straight(
    length=0.01,
    # npoints=2,
    # with_bbox=True,
    # cross_section=...
):
    return sax.reciprocal({("o1", "o2"): 1.0})

In SAX, we usually aggregate the available models in a models dictionary:

In [ ]:
factory_names = {
    "straight": gf.components.straight().factory_name,
    "bend_euler": gf.components.bend_euler().factory_name,
    "mmi1x2": gf.components.mmi1x2().factory_name,
    "mmi2x2": gf.components.mmi2x2().factory_name,
}
models = {
    factory_names["straight"]: straight,
    factory_names["bend_euler"]: bend_euler,
    factory_names["mmi1x2"]: mmi1x2,
    factory_names["mmi2x2"]: mmi2x2,
}

We can also create some dummy multimode models:

In [ ]:
def bend_euler_mm(
    angle=90.0,
    p=0.5,
    # cross_section="strip",
    # direction="ccw",
    # with_bbox=True,
    # with_arc_floorplan=True,
    # npoints=720,
):
    return sax.reciprocal(
        {
            ("o1@TE", "o2@TE"): 0.9**0.5,
            # ('o1@TE', 'o2@TM'): 0.01**0.5,
            # ('o1@TM', 'o2@TE'): 0.01**0.5,
            ("o1@TM", "o2@TM"): 0.8**0.5,
        }
    )

In [ ]:
def mmi1x2_mm(
    width=0.5,
    width_taper=1.0,
    length_taper=10.0,
    length_mmi=5.5,
    width_mmi=2.5,
    gap_mmi=0.25,
    # cross_section= strip,
    # taper= {function= taper},
    # with_bbox= True,
):
    return sax.reciprocal(
        {
            ("o1@TE", "o2@TE"): 0.45**0.5,
            ("o1@TE", "o3@TE"): 0.45**0.5,
            ("o1@TM", "o2@TM"): 0.41**0.5,
            ("o1@TM", "o3@TM"): 0.41**0.5,
            ("o1@TE", "o2@TM"): 0.01**0.5,
            ("o1@TM", "o2@TE"): 0.01**0.5,
            ("o1@TE", "o3@TM"): 0.02**0.5,
            ("o1@TM", "o3@TE"): 0.02**0.5,
        }
    )

In [ ]:
def mmi2x2_mm(
    width=0.5,
    width_taper=1.0,
    length_taper=10.0,
    length_mmi=5.5,
    width_mmi=2.5,
    gap_mmi=0.25,
    # cross_section= strip,
    # taper= {function= taper},
    # with_bbox= True,
):
    return sax.reciprocal(
        {
            ("o1@TE", "o3@TE"): 0.45**0.5,
            ("o1@TE", "o4@TE"): 1j * 0.45**0.5,
            ("o2@TE", "o3@TE"): 1j * 0.45**0.5,
            ("o2@TE", "o4@TE"): 0.45**0.5,
            ("o1@TM", "o3@TM"): 0.45**0.5,
            ("o1@TM", "o4@TM"): 1j * 0.45**0.5,
            ("o2@TM", "o3@TM"): 1j * 0.45**0.5,
            ("o2@TM", "o4@TM"): 0.45**0.5,
        }
    )

In [ ]:
def straight_mm(
    length=0.01,
    # npoints=2,
    # with_bbox=True,
    # cross_section=...
):
    return sax.reciprocal(
        {
            ("o1@TE", "o2@TE"): 1.0,
            ("o1@TM", "o2@TM"): 1.0,
        }
    )

In [ ]:
models_mm = {
    factory_names["straight"]: straight_mm,
    factory_names["bend_euler"]: bend_euler_mm,
    factory_names["mmi1x2"]: mmi1x2_mm,
    factory_names["mmi2x2"]: mmi2x2_mm,
}

We can now represent our recursive netlist model as a Directed Acyclic Graph:

In [ ]:
_, info = sax.circuit(recnet, models, top_level_name=comp.name)
dag = info.dag
draw_dag(dag)

Note that the DAG depends on the models we supply. We could for example stub one of the sub-netlists by a pre-defined model:

In [ ]:
def mzi_stub():
    return sax.reciprocal({("o1", "o2"): 1.0, ("o1", "o3"): 0.0})

_, stub_info = sax.circuit(
    recnet, {**models, mzi1_comp: mzi_stub}, top_level_name=comp.name,
)
dag2 = stub_info.dag
draw_dag(dag2, with_labels=True)

This is useful if we for example pre-calculated a certain model.

We can easily find the root of the DAG:

In [ ]:
[node for node in dag if dag.in_degree(node) == 0]

Similarly we can find the leaves:

In [ ]:
[node for node in dag if dag.out_degree(node) == 0]

To be able to simulate the circuit, we need to supply a model for each of the leaves in the dependency DAG. Let's write a validator that checks this

In [ ]:
models

In [ ]:
required = sax.get_required_circuit_models(
    recnet, models, top_level_name=comp.name,
)
assert set(required) <= set(models)

The selected child is another kfnetlist definition. Simulate it by choosing its netlist ID as the root.

In [ ]:
single_mzi, _ = sax.circuit(recnet, models, top_level_name=mzi1_comp)
single_mzi()

The resulting circuit is just another SAX model (i.e. a python function) returing an SType:

In [ ]:
?single_mzi

Let's 'execute' the circuit:

Note that we can also supply multimode models:

In [ ]:
single_mzi_mm, _ = sax.circuit(recnet, models_mm, top_level_name=mzi1_comp)
single_mzi_mm()

The full hierarchy uses the same `sax.circuit` API and a root ID from the extracted document.

single mode simulation:

In [ ]:
double_mzi, info = sax.circuit(
    recnet, models, backend="klu", top_level_name=comp.name,
)
double_mzi()

multi mode simulation:

In [ ]:
double_mzi_mm, info = sax.circuit(
    recnet, models_mm, backend="klu", return_type="SDict",
    top_level_name=comp.name,
)
double_mzi_mm()

sometimes it's useful to get the required circuit model names to be able to create the circuit:

In [ ]:
sax.get_required_circuit_models(recnet, models, top_level_name=comp.name)